## 🏭 Dataset context

This dataset contains incident reports recorded in a manufacturing plant. Each row represents a single incident logged by an operator, and includes the following information:

- **Incident metadata**: unique ID, date, time, shift (morning / afternoon / night)
- **Operator info**: name and badge number of the operator who reported the incident
- **Machine info**: machine identifier and incident severity level (1 = low, 3 = critical)
- **Incident types**: binary flags indicating the type of issue observed (overheating, pressure drop, vibration, mechanical noise, overconsumption, mechanical blockage, sensor alarm, emergency stop, quality defect)
- **Comment**: a free-text description of the incident

## 🔒 Why anonymise?

The dataset contains personal data — specifically the full names and badge numbers of plant operators. Before any analysis or sharing, this data must be anonymised to:

- Comply with GDPR and data protection regulations
- Prevent identification of individuals from the dataset
- Allow the data to be used safely for analysis and model training

Operator names are replaced with sequential identifiers (`OP_1`, `OP_2`, …) and badge numbers with (`BADGE_1`, `BADGE_2`, …). The mapping is consistent across the dataset so that per-operator analysis remains possible without exposing real identities.

## 🕵️ Operator data anonymisation

This notebook anonymises the operator data from the incident dataset before any analysis is performed.

## ⚙️ Setup

This cell initialises the output environment for the current run:

- **Timestamped directory**: a folder named after the current date and time (`YYYYMMDDHHММ`) is created under `artifacts/ingestions/incidents/`. This ensures each run produces an isolated, traceable output.
- **Graphs subdirectory**: a `graphs/` folder is created inside the timestamped directory to store all generated charts.
- **info.json**: a metadata file is written at the root of the timestamped directory, containing key statistics about the source dataset:
  - number of rows and columns read
  - number of unique machines referenced
  - total number of missing values detected

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

CSV_PATH = "artifacts/ingestions/incidents/releves_incidents.csv"

timestamp = datetime.now().strftime("%Y%m%d%H%M")
output_dir = f"artifacts/ingestions/incidents/{timestamp}"
graphs_dir = f"{output_dir}/graphs"
info_path = f"{output_dir}/info.json"
os.makedirs(graphs_dir, exist_ok=True)

df_raw = pd.read_csv(CSV_PATH)

info = {
    "timestamp": timestamp,
    "source_file": CSV_PATH,
    "num_rows": len(df_raw),
    "num_columns": len(df_raw.columns),
    "num_unique_machines": int(df_raw["machine_id"].nunique()),
    "num_missing_values": int(df_raw.isnull().sum().sum()),
}

with open(info_path, "w") as f:
    json.dump(info, f, indent=4)

print(f"Output directory : {output_dir}")
print(f"Graphs directory : {graphs_dir}")
print(f"info.json        : {info_path}")
print(json.dumps(info, indent=4))

In [ ]:
import pandas as pd

CSV_PATH = "artifacts/ingestions/incidents/releves_incidents.csv"
OUTPUT_PATH = f"{output_dir}/releves_incidents_anonymised.csv"

df = pd.read_csv(CSV_PATH)

# operator_name -> OP_1, OP_2, ...
name_map = {name: f"OP_{i+1}" for i, name in enumerate(df["operator_name"].unique())}
df["operator_name"] = df["operator_name"].map(name_map)

# operator_badge -> BADGE_1, BADGE_2, ...
badge_map = {badge: f"BADGE_{i+1}" for i, badge in enumerate(df["operator_badge"].unique())}
df["operator_badge"] = df["operator_badge"].map(badge_map)

df.to_csv(OUTPUT_PATH, index=False)

print("Name mapping:", name_map)
print("Badge mapping:", badge_map)
print(f"File generated: {OUTPUT_PATH}")
df.head(10)

## 📊 Incident distribution

This section analyses how incidents are distributed across three dimensions:

- **By day**: identifies peak days with unusually high incident counts, which may indicate specific events or operational issues.
- **By week**: provides a higher-level view of incident load over time, useful for spotting trends or recurring patterns across weeks.
- **By shift**: reveals whether certain shifts (morning, afternoon, or night) are more prone to incidents, which can inform staffing and maintenance scheduling decisions.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_plot = pd.read_csv(OUTPUT_PATH)
df_plot["date"] = pd.to_datetime(df_plot["date"])
df_plot["week"] = df_plot["date"].dt.isocalendar().week.astype(int)
df_plot["shift"] = df_plot["shift"].map({"matin": "Morning", "apres-midi": "Afternoon", "nuit": "Night"})

# By day
daily = df_plot.groupby("date").size()
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(daily.index, daily.values, width=0.6)
ax.set_title("Incident Distribution by Day")
ax.set_xlabel("Date")
ax.set_ylabel("Number of incidents")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.tight_layout()
fig.savefig(f"{graphs_dir}/incidents_by_day.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {graphs_dir}/incidents_by_day.png")

# By week
weekly = df_plot.groupby("week").size()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(weekly.index, weekly.values, width=0.6)
ax.set_title("Incident Distribution by Week")
ax.set_xlabel("Week number")
ax.set_ylabel("Number of incidents")
ax.set_xticks(weekly.index)
plt.tight_layout()
fig.savefig(f"{graphs_dir}/incidents_by_week.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {graphs_dir}/incidents_by_week.png")

# By shift
shift_order = ["Morning", "Afternoon", "Night"]
shift_counts = df_plot["shift"].value_counts().reindex(shift_order)
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(shift_counts.index, shift_counts.values, width=0.4)
ax.set_title("Incident Distribution by Shift")
ax.set_xlabel("Shift")
ax.set_ylabel("Number of incidents")
plt.tight_layout()
fig.savefig(f"{graphs_dir}/incidents_by_shift.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {graphs_dir}/incidents_by_shift.png")

## 📈 Incident type histograms

In [ ]:
import matplotlib.pyplot as plt

TYPE_LABELS = {
    "type_surchauffe": "Overheating",
    "type_baisse_pression": "Pressure drop",
    "type_vibration": "Vibration",
    "type_bruit_mecanique": "Mechanical noise",
    "type_surconsommation": "Overconsumption",
    "type_blocage_mecanique": "Mechanical blockage",
    "type_alarme_capteur": "Sensor alarm",
    "type_arret_urgence": "Emergency stop",
    "type_defaut_qualite": "Quality defect",
}

df_plot = pd.read_csv(OUTPUT_PATH)
type_cols = [col for col in df_plot.columns if col.startswith("type_")]

for col in type_cols:
    machine_counts = df_plot.groupby("machine_id")[col].sum().sort_values(ascending=False)
    label = TYPE_LABELS.get(col, col)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(machine_counts.index, machine_counts.values, edgecolor="black")
    ax.set_title(f"Incidents by Machine — {label}")
    ax.set_xlabel("Machine")
    ax.set_ylabel(f"Number of {label.lower()} incidents")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    fig.savefig(f"{graphs_dir}/{col}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {graphs_dir}/{col}.png")

## 🔗 Severity vs incident type correlation

In [ ]:
import matplotlib.pyplot as plt

TYPE_LABELS = {
    "type_surchauffe": "Overheating",
    "type_baisse_pression": "Pressure drop",
    "type_vibration": "Vibration",
    "type_bruit_mecanique": "Mechanical noise",
    "type_surconsommation": "Overconsumption",
    "type_blocage_mecanique": "Mechanical blockage",
    "type_alarme_capteur": "Sensor alarm",
    "type_arret_urgence": "Emergency stop",
    "type_defaut_qualite": "Quality defect",
}

df_plot = pd.read_csv(OUTPUT_PATH)
type_cols = [col for col in df_plot.columns if col.startswith("type_")]

correlations = df_plot[type_cols].corrwith(df_plot["severity"]).rename(TYPE_LABELS)
correlations = correlations.sort_values()

colors = ["#d73027" if v >= 0 else "#4575b4" for v in correlations.values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(correlations.index, correlations.values, color=colors, edgecolor="black")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Pearson Correlation — Severity vs Incident Type")
ax.set_xlabel("Correlation coefficient")
ax.set_ylabel("Incident type")
plt.tight_layout()
fig.savefig(f"{graphs_dir}/severity_type_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {graphs_dir}/severity_type_correlation.png")